# O Que as Pessoas Dizem? Análise de Respostas da Comunidade
## Como as Audiências Reagem a Trump vs AOC no Twitter/X

Quando políticos postam nas redes sociais, as pessoas respondem -- às vezes com apoio, às vezes com críticas, às vezes com humor ou indignação. Essas respostas (chamadas de **respostas** ou **comentários**) nos dizem muito sobre como os cidadãos participam das discussões políticas online.

Neste notebook, analisamos os comentários que as pessoas deixam nos tweets de Trump e AOC. Fazemos perguntas simples mas importantes:
- **Quantas** pessoas respondem a cada político?
- **Quão rápido** elas respondem?
- **Quem** são esses comentaristas? São pessoas comuns ou contas de alto perfil?
- **Qual tom** os comentários têm -- positivo, negativo ou neutro?
- **Quais palavras e tópicos** aparecem com mais frequência nos comentários?

Não é necessário ter conhecimento de programação ou estatística para acompanhar esta análise. Explicamos cada conceito conforme ele aparece.

---

**Termos-chave que você verá neste notebook:**
- **Resposta / Comentário**: Uma reação que alguém posta debaixo de um tweet
- **Média**: Some todos os números e divida pela quantidade deles. Por exemplo, se três tweets receberam 10, 20 e 30 respostas, a média é (10+20+30) / 3 = 20 respostas.
- **Mediana (valor do meio)**: Se você organizar todos os números do menor para o maior, a mediana é aquele que fica bem no meio. Diferente da média, a mediana não é puxada por valores extremos.
- **Distribuição**: Uma forma de mostrar como os valores estão espalhados -- por exemplo, a maioria dos tweets recebe poucas respostas, ou as contagens de respostas variam enormemente?

In [ ]:
# --- Configuração: Carregando as ferramentas e dados que precisamos ---
# (Esta célula carrega as bibliotecas de software e estilização. Você pode ignorar os detalhes técnicos.)

import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from utils.data_loader import load_tweets, load_replies
from utils.plot_helpers import (
    setup_style, get_colors, get_user_label,
    comparative_bar, comparative_boxplot, comparative_hist,
    format_large_numbers, add_summary_stats, set_language
)
from utils.text_helpers import get_word_frequencies, extract_hashtags
from config import FIGURE_SIZE, FIGURE_SIZE_SMALL, FIGURE_SIZE_LARGE, RANDOM_SEED

setup_style()
set_language('pt-br')
colors = get_colors()
np.random.seed(RANDOM_SEED)

print('Configuração concluída.')

In [ ]:
# --- Carregar os dados ---
# Carregamos dois datasets:
#   1. Os tweets originais postados por Trump e AOC
#   2. As respostas (comentários) que outras pessoas deixaram nesses tweets

df = load_tweets()
replies_df = load_replies()

print(f"Total de tweets coletados: {len(df)}")
print(f"Total de respostas (comentários) coletadas: {len(replies_df)}")
print(f"\nTweets por político: {df['user'].value_counts().to_dict()}")
print(f"Respostas por político: {replies_df['parent_user'].value_counts().to_dict()}")

---
## 1. Quantas Pessoas Respondem? Visão Geral do Volume de Respostas

A pergunta mais básica que podemos fazer é: **quanta conversa cada político gera?**

Quando alguém posta um tweet e recebe milhares de respostas, significa que o conteúdo tocou um nervo -- as pessoas se sentiram compelidas a responder. O número de respostas é uma das formas mais simples de medir o quanto a mensagem de um político gera discussão.

Abaixo, mostramos três gráficos lado a lado:
1. **Total de respostas coletadas** -- a contagem bruta de todos os comentários que coletamos para cada político
2. **Média de respostas por tweet** -- em um tweet típico, quantos comentários cada político recebe?
3. **Distribuição das contagens de respostas** -- a maioria dos tweets recebe um número similar de respostas, ou alguns tweets explodem com comentários enquanto outros recebem muito poucos?

In [ ]:
# --- Visão Geral do Volume de Respostas ---
fig, axes = plt.subplots(1, 3, figsize=FIGURE_SIZE_LARGE)

# 1. Total de respostas por usuário
ax = axes[0]
users = ['trump', 'aoc']
total_replies = [len(replies_df[replies_df['parent_user'] == u]) for u in users]
user_labels = [get_user_label(u) for u in users]

bars = ax.bar(user_labels, total_replies,
              color=[colors[u] for u in users], width=0.5, edgecolor='white')
for bar, val in zip(bars, total_replies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"{val:,}", ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Total de Comentários Coletados', fontweight='bold')
ax.set_ylabel('Número de Comentários')
ax.set_ylim(0, max(total_replies) * 1.15 if max(total_replies) > 0 else 1)
format_large_numbers(ax)

# 2. Média de respostas por tweet (do reply_count do df de tweets)
ax = axes[1]
avg_replies = [df[df['user'] == u]['reply_count'].mean() for u in users]

bars = ax.bar(user_labels, avg_replies,
              color=[colors[u] for u in users], width=0.5, edgecolor='white')
for bar, val in zip(bars, avg_replies):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"{val:,.0f}", ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Média de Comentários por Tweet', fontweight='bold')
ax.set_ylabel('Média de Comentários')
ax.set_ylim(0, max(avg_replies) * 1.15 if max(avg_replies) > 0 else 1)
format_large_numbers(ax)

# 3. Histograma da distribuição de reply_count (do df de tweets)
ax = axes[2]
for user in users:
    user_data = df[df['user'] == user]['reply_count'].dropna()
    ax.hist(user_data, bins=20, alpha=0.5, label=get_user_label(user),
            color=colors[user], edgecolor='white')
ax.set_title('Como as Contagens de Respostas se Distribuem', fontweight='bold')
ax.set_xlabel('Número de Comentários em um Único Tweet')
ax.set_ylabel('Quantos Tweets Receberam Essa Quantidade')
ax.legend(fontsize=9)

fig.suptitle('Volume de Respostas: Quanta Conversa Cada Político Gera?',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### O que isso nos diz?

Estes gráficos revelam a **escala de conversa pública** que cada político gera. Uma contagem total de respostas mais alta significa que mais pessoas sentem necessidade de responder. A média de respostas por tweet nos diz como é um tweet "normal" para cada político. E a distribuição (o terceiro gráfico) nos diz se o engajamento é consistente ou impulsionado por alguns momentos virais.

Se as contagens de respostas de um político estão agrupadas próximas, sua audiência responde de forma bastante consistente. Se o gráfico está muito espalhado, significa que alguns tweets mal são notados enquanto outros geram discussões massivas.

In [ ]:
# --- Top 10 tweets que receberam mais comentários ---
# Quais tweets específicos geraram mais discussão?

for user in ['trump', 'aoc']:
    user_df = df[df['user'] == user].nlargest(10, 'reply_count')
    
    display_df = user_df[['text', 'reply_count', 'favorite_count']].copy()
    display_df['text'] = display_df['text'].str[:90] + '...'
    display_df = display_df.reset_index(drop=True)
    display_df.index = display_df.index + 1
    display_df.index.name = 'Posição'
    
    display_df['reply_count'] = display_df['reply_count'].apply(lambda x: f"{x:,.0f}")
    display_df['favorite_count'] = display_df['favorite_count'].apply(lambda x: f"{x:,.0f}")
    display_df.columns = ['Texto do Tweet (primeiros 90 caracteres)', 'Comentários', 'Curtidas']
    
    print(f"\n{'='*90}")
    print(f"Top 10 Tweets Mais Comentados -- {get_user_label(user)}")
    print(f"{'='*90}")
    display(display_df)

### O que isso nos diz?

Olhar os tweets específicos que receberam mais comentários nos ajuda a entender **quais tópicos geram mais discussão**. São sobre políticas públicas? Ataques pessoais? Notícias de última hora? O conteúdo desses tweets mais comentados revela o que a audiência se importa mais intensamente.

Note se há uma relação entre curtidas e comentários -- um tweet com muitas curtidas mas poucos comentários pode ser algo com que as pessoas concordam silenciosamente, enquanto um tweet com muitos comentários sugere debate ativo.

---
## 2. Quão Rápido as Pessoas Respondem? Velocidade das Respostas

A velocidade importa na discussão política online. Quando um político tuíta algo controverso ou importante, **quão rápido as pessoas entram para comentar?**

O **tempo de resposta** é simplesmente quantos minutos passam entre quando um tweet é postado e quando um comentário aparece. Se a maioria das respostas chega em minutos, sugere que a audiência está monitorando ativamente o feed do político em tempo real -- talvez via notificações ou verificando a página frequentemente.

**Novo termo -- Box plot (diagrama de caixa)**: Um box plot é um gráfico simples que mostra como os dados estão distribuídos. A caixa no meio mostra onde os 50% centrais dos valores se encontram. A linha dentro da caixa é a **mediana** (valor do meio). Os "bigodes" (linhas que se estendem da caixa) mostram a faixa mais ampla, e pontos além dos bigodes são valores atípicos incomuns.

In [ ]:
# --- Quão rápido as pessoas respondem? ---
# Limitamos o atraso a 24 horas (1.440 minutos) para que os gráficos sejam mais fáceis de ler.
# Algumas respostas chegam dias depois, mas focamos nas primeiras 24 horas.

replies_capped = replies_df.copy()
replies_capped['reply_delay_capped'] = replies_capped['reply_delay_minutes'].clip(upper=1440)

fig, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE_LARGE)

# 1. Como os tempos de resposta se distribuem
ax = axes[0]
for user in ['trump', 'aoc']:
    user_data = replies_capped[replies_capped['parent_user'] == user]['reply_delay_capped'].dropna()
    ax.hist(user_data, bins=50, alpha=0.5, label=get_user_label(user),
            color=colors[user], edgecolor='white')
ax.set_title('Quanto Tempo Até as Pessoas Responderem (primeiras 24 horas)', fontweight='bold')
ax.set_xlabel('Minutos Após o Tweet Ser Postado')
ax.set_ylabel('Número de Respostas')
ax.legend(fontsize=9)

# 2. Box plot do tempo de resposta
ax = axes[1]
users = ['trump', 'aoc']
data = [replies_capped[replies_capped['parent_user'] == u]['reply_delay_capped'].dropna().values
        for u in users]
user_labels = [get_user_label(u) for u in users]

bp = ax.boxplot(data, labels=user_labels, patch_artist=True, widths=0.5,
                medianprops={'color': 'black', 'linewidth': 1.5})
for patch, user in zip(bp['boxes'], users):
    patch.set_facecolor(colors[user])
    patch.set_alpha(0.7)
ax.set_title('Resumo do Tempo de Resposta (Box Plot, primeiras 24 horas)', fontweight='bold')
ax.set_ylabel('Minutos Após o Tweet Ser Postado')

fig.suptitle('Velocidade de Resposta: Quão Rápido as Audiências Reagem?',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Imprimir o tempo típico (mediana) de resposta
print("\nTempo típico (mediana) até uma resposta aparecer:")
for user in ['trump', 'aoc']:
    median_delay = replies_df[replies_df['parent_user'] == user]['reply_delay_minutes'].median()
    print(f"  {get_user_label(user)}: {median_delay:.1f} minutos ({median_delay/60:.1f} horas)")

### O que isso nos diz?

Os gráficos de tempo de resposta revelam **quão atenta a audiência de cada político é**. Se a maioria das respostas chega nos primeiros minutos, significa que as pessoas estão acompanhando de perto -- podem ter as notificações ligadas ou verificam a conta frequentemente.

Um tempo típico de resposta mais curto sugere uma audiência mais engajada e em tempo real. Um atraso mais longo sugere uma audiência mais casual que encontra os tweets navegando em vez de monitorar ativamente.

O box plot ajuda a comparar rapidamente: uma caixa mais baixa significa respostas mais rápidas no geral.

In [ ]:
# --- A velocidade de resposta mudou ao longo do tempo? ---
# Para cada tweet, calculamos o tempo típico (mediana) de resposta,
# depois plotamos isso ao longo do período coletado.

fig, ax = plt.subplots(figsize=FIGURE_SIZE)

for user in ['trump', 'aoc']:
    # Obter datas de criação dos tweets
    user_tweets = df[df['user'] == user][['tweet_id', 'created_at']].copy()
    user_tweets = user_tweets.sort_values('created_at')
    
    # Calcular mediana do tempo de resposta por tweet
    user_replies = replies_df[replies_df['parent_user'] == user]
    median_delay_per_tweet = user_replies.groupby('parent_tweet_id')['reply_delay_minutes'].median()
    median_delay_per_tweet = median_delay_per_tweet.reset_index()
    median_delay_per_tweet.columns = ['tweet_id', 'median_delay']
    
    # Mesclar com datas de criação dos tweets
    merged = user_tweets.merge(median_delay_per_tweet, on='tweet_id', how='inner')
    merged = merged.sort_values('created_at')
    
    if len(merged) > 0:
        ax.plot(merged['created_at'], merged['median_delay'],
                marker='o', markersize=3, label=get_user_label(user),
                color=colors[user], linewidth=1, alpha=0.7)

ax.set_title('Como a Velocidade de Resposta Mudou ao Longo do Tempo', fontweight='bold', fontsize=14)
ax.set_xlabel('Data em Que o Tweet Foi Postado')
ax.set_ylabel('Tempo Típico de Resposta (minutos)')
ax.legend(fontsize=9)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### O que isso nos diz?

Este gráfico temporal mostra se a atenção da audiência tem sido **consistente ou se alterando**. Se a linha tende para baixo ao longo do tempo, as pessoas estão respondendo mais rápido (a audiência pode estar ficando mais engajada). Se tende para cima, os tempos de resposta estão desacelerando.

Picos ou quedas repentinos frequentemente correspondem a grandes eventos -- uma declaração controversa, um momento viral, ou um ciclo de notícias que atrai mais (ou menos) olhares para a conta daquele político.

---
## 3. Quem Responde? O Perfil dos Comentaristas

Nem todos os comentaristas são iguais. Alguns são cidadãos comuns com um punhado de seguidores, enquanto outros são contas de alto perfil com milhões de seguidores. Entender **quem** responde nos diz sobre a natureza da comunidade online de cada político.

Analisamos três características:
1. **Quantos seguidores** os comentaristas têm? Isso nos diz se a conversa é movida por pessoas comuns ou por contas influentes.
2. **Os comentaristas são verificados?** No Twitter/X, há dois tipos de verificação:
   - **Verificado Legado** (a antiga marca azul) -- dada a figuras públicas notáveis, jornalistas, organizações
   - **Verificado Azul (X Premium)** -- disponível para qualquer pessoa que pague pela assinatura
3. **Quem comenta mais?** Algumas pessoas respondem aos tweets de um político repetidamente. Esses "super-comentaristas" podem ser apoiadores dedicados, críticos persistentes ou contas automatizadas.

**Novo termo -- Escala log**: Quando números variam de muito pequenos (como 1 seguidor) a muito grandes (como 10 milhões de seguidores), um gráfico normal comprimiria todos os valores pequenos juntos. Uma **escala log** distribui as coisas para que possamos ver toda a faixa. Cada passo na escala representa um aumento de 10x (1, 10, 100, 1.000, etc.).

In [ ]:
# --- Quantos seguidores os comentaristas têm? ---
fig, ax = plt.subplots(figsize=FIGURE_SIZE)

for user in ['trump', 'aoc']:
    user_data = replies_df[replies_df['parent_user'] == user]['user_followers_count'].dropna()
    # Filtrar para valores positivos para escala log
    user_data = user_data[user_data > 0]
    ax.hist(np.log10(user_data), bins=40, alpha=0.5, label=get_user_label(user),
            color=colors[user], edgecolor='white')

ax.set_title('Quantos Seguidores os Comentaristas Têm?', fontweight='bold', fontsize=14)
ax.set_xlabel('Contagem de Seguidores (escala log -- cada passo é 10x mais)')
ax.set_ylabel('Número de Comentaristas')
ax.legend(fontsize=9)

# Adicionar rótulos legíveis mostrando contagens reais de seguidores
tick_positions = [0, 1, 2, 3, 4, 5, 6, 7]
tick_labels = ['1', '10', '100', '1K', '10K', '100K', '1M', '10M']
ax.set_xticks(tick_positions)
ax.set_xticklabels(tick_labels)

plt.tight_layout()
plt.show()

# Imprimir estatísticas resumidas
print("\nResumo da Contagem de Seguidores dos Comentaristas:")
print("(Média = some tudo e divida; Mediana = o valor do meio)")
for user in ['trump', 'aoc']:
    user_data = replies_df[replies_df['parent_user'] == user]['user_followers_count'].dropna()
    label = get_user_label(user)
    print(f"\n  {label}:")
    print(f"    Média:   {user_data.mean():,.0f} seguidores")
    print(f"    Mediana: {user_data.median():,.0f} seguidores")
    print(f"    Maior:   {user_data.max():,.0f} seguidores")

### O que isso nos diz?

A distribuição de seguidores revela se os comentários vêm principalmente de **usuários comuns** (contas com dezenas ou centenas de seguidores) ou de **contas influentes** (milhares ou milhões de seguidores).

Preste atenção na diferença entre a média e a mediana. Se a média é muito maior que a mediana, significa que algumas contas de muitos seguidores estão puxando a média para cima, mas a maioria dos comentaristas são usuários comuns. Isso é comum em discussões políticas -- um pequeno número de vozes proeminentes participa junto com uma grande base de cidadãos comuns.

In [ ]:
# --- Os comentaristas são verificados? ---
fig, ax = plt.subplots(figsize=FIGURE_SIZE)

users = ['trump', 'aoc']
verification_types = ['Verificado Legado\n(Figuras notáveis)', 'Verificado Azul\n(Assinantes X Premium)']
x = np.arange(len(verification_types))
width = 0.35

for i, user in enumerate(users):
    user_replies = replies_df[replies_df['parent_user'] == user]
    n_total = len(user_replies)
    
    pct_verified = (user_replies['user_is_verified'].sum() / n_total * 100) if n_total > 0 else 0
    pct_blue = (user_replies['user_is_blue_verified'].sum() / n_total * 100) if n_total > 0 else 0
    
    values = [pct_verified, pct_blue]
    offset = -width / 2 + i * width
    bars = ax.bar(x + offset, values, width, label=get_user_label(user),
                  color=colors[user], alpha=0.8, edgecolor='white')
    
    for bar, val in zip(bars, values):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                    f"{val:.1f}%", ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Qual Porcentagem dos Comentaristas É Verificada?', fontweight='bold', fontsize=14)
ax.set_ylabel('Porcentagem de Comentaristas (%)')
ax.set_xticks(x)
ax.set_xticklabels(verification_types)
ax.legend(fontsize=10)
ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
plt.tight_layout()
plt.show()

### O que isso nos diz?

O status de verificação nos dá uma noção aproximada de **quem compõe a comunidade de comentaristas de cada político**. Uma porcentagem maior de comentaristas Verificados Legados sugere que jornalistas, figuras públicas e organizações estão participando da conversa. Uma porcentagem maior de Verificados Azuis (X Premium) indica que assinantes pagos -- que podem ser usuários mais ativos ou investidos -- participam mais.

Diferenças entre os dois políticos podem refletir os tipos de audiências que atraem: um pode atrair mais atenção institucional/midiática enquanto o outro engaja uma audiência mais de base.

In [ ]:
# --- Quem comenta mais? Os "super-comentaristas" ---
# Estas são contas que responderam aos tweets do mesmo político muitas vezes.

for user in ['trump', 'aoc']:
    user_replies = replies_df[replies_df['parent_user'] == user]
    
    # Contar quantas vezes cada pessoa comentou
    replier_stats = user_replies.groupby('user_name').agg(
        reply_count=('reply_id', 'count'),
        follower_count=('user_followers_count', 'max')
    ).sort_values('reply_count', ascending=False).head(10)
    
    replier_stats = replier_stats.reset_index()
    replier_stats.index = replier_stats.index + 1
    replier_stats.index.name = 'Posição'
    
    replier_stats['follower_count'] = replier_stats['follower_count'].apply(lambda x: f"{x:,.0f}")
    replier_stats.columns = ['Nome de Usuário', 'Número de Comentários', 'Seguidores']
    
    print(f"\n{'='*70}")
    print(f"Top 10 Comentaristas Mais Frequentes -- {get_user_label(user)}")
    print(f"{'='*70}")
    display(replier_stats)

### O que isso nos diz?

Os "super-comentaristas" são pessoas que respondem repetidamente aos tweets de um político. Eles podem ser:
- **Apoiadores dedicados** que consistentemente aparecem para defender seu político
- **Críticos persistentes** que se opõem a tudo que o político diz
- **Contas bot ou automatizadas** programadas para responder automaticamente
- **Contas ativistas** tentando amplificar uma mensagem particular

Olhar as contagens de seguidores desses principais comentaristas nos ajuda a entender se as vozes mais ativas são contas influentes ou contas menores com participação desproporcional.

---
## 4. Os Comentários São Positivos ou Negativos? Análise de Sentimento

Além de contar comentários, queremos entender o **tom emocional** da conversa. As pessoas estão respondendo com apoio e elogios, ou com críticas e raiva?

**Como medimos isso (análise de sentimento explicada de forma simples):**
Usamos um computador para verificar se os comentários soam positivos, negativos ou neutros, procurando palavras-chave. Por exemplo:
- Palavras como **"great," "love," "amazing," "thank you"** sugerem um tom **positivo**
- Palavras como **"terrible," "hate," "stupid," "liar"** sugerem um tom **negativo**
- Se um comentário tem aproximadamente a mesma quantidade de palavras positivas e negativas, ou nenhuma palavra forte, é classificado como **neutro**

Esta é uma abordagem simplificada -- não consegue detectar sarcasmo ou significados sutis -- mas nos dá uma visão geral útil das reações emocionais da comunidade.

In [ ]:
# --- Qual é o tom geral dos comentários? ---
fig, ax = plt.subplots(figsize=FIGURE_SIZE)

sentiment_order = ['positive', 'neutral', 'negative']
sentiment_labels = {'positive': 'Positivo', 'neutral': 'Neutro', 'negative': 'Negativo'}
sentiment_colors = {'positive': '#2a9d8f', 'neutral': '#e9c46a', 'negative': '#e76f51'}

users = ['trump', 'aoc']
proportions = {}

for user in users:
    user_replies = replies_df[replies_df['parent_user'] == user]
    counts = user_replies['sentiment_simple'].value_counts()
    total = counts.sum()
    proportions[user] = {s: (counts.get(s, 0) / total * 100) if total > 0 else 0
                         for s in sentiment_order}

# Gráfico de barras empilhadas
x = np.arange(len(users))
width = 0.5
bottom = np.zeros(len(users))

for sentiment in sentiment_order:
    values = [proportions[u][sentiment] for u in users]
    bars = ax.bar(x, values, width, bottom=bottom,
                  label=sentiment_labels[sentiment], color=sentiment_colors[sentiment],
                  edgecolor='white', linewidth=0.5)
    
    # Adicionar rótulos de porcentagem no meio de cada segmento
    for j, (bar, val) in enumerate(zip(bars, values)):
        if val > 3:  # Só rotular segmentos > 3% para legibilidade
            ax.text(bar.get_x() + bar.get_width() / 2,
                    bottom[j] + val / 2,
                    f"{val:.1f}%", ha='center', va='center',
                    fontsize=10, fontweight='bold', color='white')
    bottom += values

ax.set_title('Qual É o Tom dos Comentários? (Positivo, Neutro ou Negativo)',
             fontweight='bold', fontsize=14)
ax.set_ylabel('Porcentagem de Comentários (%)')
ax.set_xticks(x)
ax.set_xticklabels([get_user_label(u) for u in users])
ax.legend(loc='upper right', fontsize=10)
ax.set_ylim(0, 105)
plt.tight_layout()
plt.show()

# Imprimir porcentagens exatas
print("\nPorcentagens Exatas:")
for user in users:
    print(f"\n  {get_user_label(user)}:")
    for sentiment in sentiment_order:
        print(f"    {sentiment_labels[sentiment]:10s}: {proportions[user][sentiment]:.2f}%")

### O que isso nos diz?

O detalhamento do sentimento revela a **atmosfera emocional** da seção de comentários de cada político. Uma proporção maior de comentários negativos sugere que o político enfrenta mais oposição ou controvérsia em suas respostas. Uma proporção maior de comentários positivos sugere uma audiência mais apoiadora.

É importante notar que o Twitter/X político tende a atrair opiniões fortes, então ambos os políticos podem ter sentimento negativo substancial. A comparação chave é se a seção de comentários de um político é significativamente mais hostil ou apoiadora que a do outro.

Lembre-se: neutro não significa "indiferente" -- pode incluir respostas factuais, perguntas, ou comentários que simplesmente não contêm palavras-chave fortemente positivas ou negativas.

In [ ]:
# --- Comentários positivos ou negativos recebem mais curtidas? ---
# Isso nos diz que tipo de tom a audiência mais ampla recompensa.

fig, ax = plt.subplots(figsize=FIGURE_SIZE)

sentiment_order = ['positive', 'neutral', 'negative']
sentiment_labels = {'positive': 'Positivo', 'neutral': 'Neutro', 'negative': 'Negativo'}
users = ['trump', 'aoc']
x = np.arange(len(sentiment_order))
width = 0.35

for i, user in enumerate(users):
    user_replies = replies_df[replies_df['parent_user'] == user]
    means = []
    for sentiment in sentiment_order:
        subset = user_replies[user_replies['sentiment_simple'] == sentiment]
        means.append(subset['favorite_count'].mean() if len(subset) > 0 else 0)
    
    offset = -width / 2 + i * width
    bars = ax.bar(x + offset, means, width, label=get_user_label(user),
                  color=colors[user], alpha=0.8, edgecolor='white')
    
    for bar, val in zip(bars, means):
        if val > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                    f"{val:,.0f}", ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_title('Qual Tipo de Comentário Recebe Mais Curtidas?', fontweight='bold', fontsize=14)
ax.set_ylabel('Média de Curtidas')
ax.set_xticks(x)
ax.set_xticklabels([sentiment_labels[s] for s in sentiment_order])
ax.legend(fontsize=10)
ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
format_large_numbers(ax)
plt.tight_layout()
plt.show()

### O que isso nos diz?

Este gráfico revela **o que a audiência mais ampla recompensa**. Se comentários negativos recebem mais curtidas, sugere que a audiência (ou pelo menos a maioria das pessoas vendo aquelas respostas) é mais oposicionista. Se comentários positivos recebem mais curtidas, a audiência é mais apoiadora.

Esta é uma distinção importante: o tom dos comentários nos diz o que as pessoas *escrevem*, mas as curtidas nesses comentários nos dizem com o que a *audiência mais ampla concorda*. Essas duas coisas podem contar histórias muito diferentes.

In [ ]:
# --- Comentários positivos e negativos mais curtidos ---
# Estes são os comentários específicos que mais ressoaram com cada audiência.

for user in ['trump', 'aoc']:
    user_replies = replies_df[replies_df['parent_user'] == user]
    label = get_user_label(user)
    
    sentiment_labels_map = {'positive': 'Positivos', 'negative': 'Negativos'}
    
    for sentiment in ['positive', 'negative']:
        subset = user_replies[user_replies['sentiment_simple'] == sentiment]
        top5 = subset.nlargest(5, 'favorite_count')
        
        display_df = top5[['user_name', 'text', 'favorite_count']].copy()
        display_df['text'] = display_df['text'].str[:100] + '...'
        display_df['favorite_count'] = display_df['favorite_count'].apply(lambda x: f"{x:,.0f}")
        display_df = display_df.reset_index(drop=True)
        display_df.index = display_df.index + 1
        display_df.index.name = 'Posição'
        display_df.columns = ['Nome de Usuário', 'Texto do Comentário (primeiros 100 caracteres)', 'Curtidas']
        
        print(f"\n{'='*90}")
        print(f"Top 5 Comentários {sentiment_labels_map[sentiment]} Mais Curtidos -- {label}")
        print(f"{'='*90}")
        display(display_df)

### O que isso nos diz?

Ler o texto real dos comentários mais populares nos dá **insight qualitativo** que números sozinhos não podem fornecer. Estas são as vozes que mais ressoaram com a audiência.

Pergunte a si mesmo: Os comentários positivos mais populares expressam admiração genuína, ou são torcida partidária? Os comentários negativos mais populares são críticas substantivas, ou ataques pessoais? A natureza desses principais comentários revela a qualidade e o caráter do discurso político em cada comunidade.

---
## 5. Engajamento Dentro dos Comentários

Quando alguém responde ao tweet de um político, essa resposta pode receber curtidas, repostagens e visualizações. Isso é o que chamamos de **engajamento dentro dos comentários** -- a conversa que acontece *sobre a conversa*.

Quando uma resposta recebe muitas curtidas, significa que a audiência considerou aquela resposta valiosa, engraçada ou concordou com ela. Alto engajamento dentro dos comentários sinaliza que a discussão nas threads de resposta tem sua própria audiência e importância -- não é apenas ruído de fundo, mas uma arena ativa de discussão política.

In [ ]:
# --- Quanta atenção os próprios comentários recebem? ---
fig, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE_LARGE)

# 1. Média de curtidas por comentário
ax = axes[0]
users = ['trump', 'aoc']
mean_fav = [replies_df[replies_df['parent_user'] == u]['favorite_count'].mean() for u in users]
user_labels = [get_user_label(u) for u in users]

bars = ax.bar(user_labels, mean_fav,
              color=[colors[u] for u in users], width=0.5, edgecolor='white')
for bar, val in zip(bars, mean_fav):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"{val:,.1f}", ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_title('Média de Curtidas por Comentário', fontweight='bold')
ax.set_ylabel('Média de Curtidas')
ax.set_ylim(0, max(mean_fav) * 1.15 if max(mean_fav) > 0 else 1)
format_large_numbers(ax)

# 2. Como as curtidas dos comentários se distribuem?
ax = axes[1]
for user in users:
    user_data = replies_df[replies_df['parent_user'] == user]['favorite_count'].dropna()
    # Usar escala log para melhor visualização (maioria dos comentários recebe poucas curtidas, alguns recebem muitas)
    user_data_log = user_data[user_data > 0]
    if len(user_data_log) > 0:
        ax.hist(np.log10(user_data_log + 1), bins=30, alpha=0.5,
                label=get_user_label(user), color=colors[user], edgecolor='white')

ax.set_title('Como as Curtidas dos Comentários se Distribuem (Escala Log)', fontweight='bold')
ax.set_xlabel('Número de Curtidas (escala log)')
ax.set_ylabel('Número de Comentários')
ax.legend(fontsize=9)

fig.suptitle('Engajamento Dentro dos Comentários: As Respostas Também Recebem Atenção?',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Comentário mais curtido de cada político
print("\nComentário Mais Curtido de Cada Político:")
for user in ['trump', 'aoc']:
    user_replies = replies_df[replies_df['parent_user'] == user]
    if len(user_replies) > 0:
        top_reply = user_replies.loc[user_replies['favorite_count'].idxmax()]
        print(f"\n  {get_user_label(user)}:")
        print(f"    Por: @{top_reply['user_name']}")
        print(f"    Curtidas: {top_reply['favorite_count']:,.0f}")
        print(f"    Repostagens: {top_reply['retweet_count']:,.0f}")
        print(f"    Texto: {str(top_reply['text'])[:150]}...")

### O que isso nos diz?

Quando os próprios comentários recebem muitas curtidas, significa que a seção de respostas se tornou **uma conversa por si só** -- não apenas um espaço para respostas unilaterais, mas um fórum ativo onde as pessoas se engajam com as ideias umas das outras.

Maior engajamento dentro dos comentários sugere uma comunidade mais vibrante e participativa. Também significa que vozes individuais na seção de respostas podem alcançar grandes audiências, amplificando certas perspectivas e moldando a narrativa em torno dos tweets de um político.

---
## 6. Sobre o Que as Pessoas Estão Falando? Palavras e Hashtags nos Comentários

Além do tom, queremos saber **quais tópicos e temas** dominam as seções de comentários. Ao analisar as palavras e hashtags mais comuns usadas nas respostas, podemos identificar as preocupações, slogans e ideias que a audiência de cada político mais se importa.

In [ ]:
# --- Quais são as palavras mais comuns nos comentários? ---
fig, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE_LARGE)

for idx, user in enumerate(['trump', 'aoc']):
    ax = axes[idx]
    user_replies = replies_df[replies_df['parent_user'] == user]
    texts = user_replies['text'].dropna().tolist()
    
    # Obter as 20 palavras mais comuns (excluindo palavras comuns como "the", "and", etc.)
    word_freqs = get_word_frequencies(texts, top_n=20)
    
    if word_freqs:
        words = [w for w, c in word_freqs]
        counts = [c for w, c in word_freqs]
        
        y_pos = np.arange(len(words))
        ax.barh(y_pos, counts, color=colors[user], alpha=0.8, edgecolor='white')
        ax.set_yticks(y_pos)
        ax.set_yticklabels(words, fontsize=9)
        ax.invert_yaxis()
        ax.set_xlabel('Quantas Vezes Esta Palavra Aparece')
    
    ax.set_title(f'{get_user_label(user)}', fontweight='bold', fontsize=12)

fig.suptitle('Top 20 Palavras Mais Comuns nos Comentários', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### O que isso nos diz?

As palavras mais comuns revelam **quais tópicos dominam cada conversa**. Procure por:
- **Nomes** (do político, oponentes ou figuras públicas) -- mostrando em quem a audiência está focada
- **Palavras relacionadas a políticas** -- mostrando quais questões importam para a audiência
- **Palavras emocionais** -- mostrando o humor geral
- **Palavras de ação** ("vote," "fight," "support") -- mostrando se a audiência está sendo mobilizada

Comparar os dois lado a lado nos diz se essas duas comunidades políticas estão falando sobre as mesmas coisas ou vivendo em mundos de informação completamente diferentes.

In [ ]:
# --- Quais hashtags os comentaristas usam? ---
# Hashtags (palavras começando com #) são rótulos que as pessoas usam para conectar seu comentário
# a um tópico ou movimento mais amplo.

fig, axes = plt.subplots(1, 2, figsize=FIGURE_SIZE_LARGE)

for idx, user in enumerate(['trump', 'aoc']):
    ax = axes[idx]
    user_replies = replies_df[replies_df['parent_user'] == user]
    
    # Extrair todas as hashtags dos textos dos comentários
    all_hashtags = []
    for text in user_replies['text'].dropna():
        all_hashtags.extend([h.lower() for h in extract_hashtags(text)])
    
    # Contar e obter top 10
    if all_hashtags:
        hashtag_counts = pd.Series(all_hashtags).value_counts().head(10)
        
        y_pos = np.arange(len(hashtag_counts))
        ax.barh(y_pos, hashtag_counts.values, color=colors[user], alpha=0.8, edgecolor='white')
        ax.set_yticks(y_pos)
        ax.set_yticklabels([f'#{h}' for h in hashtag_counts.index], fontsize=9)
        ax.invert_yaxis()
        ax.set_xlabel('Quantas Vezes Esta Hashtag Aparece')
    else:
        ax.text(0.5, 0.5, 'Nenhuma hashtag encontrada', transform=ax.transAxes,
                ha='center', va='center', fontsize=12, color='gray')
    
    ax.set_title(f'{get_user_label(user)}', fontweight='bold', fontsize=12)

fig.suptitle('Top 10 Hashtags Usadas nos Comentários', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### O que isso nos diz?

Hashtags são como **bandeiras ou slogans** que os comentaristas levantam. Elas conectam respostas individuais a movimentos políticos maiores, campanhas ou questões. As hashtags mais populares revelam:
- **Filiações políticas** (hashtags relacionadas a partidos)
- **Campanhas temáticas** (hashtags sobre políticas ou causas específicas)
- **Mensagens de oposição** (hashtags usadas para criticar ou zombar)
- **Momentos virais** (hashtags ligadas a eventos atuais)

Se as seções de comentários de ambos os políticos compartilham muitas das mesmas hashtags, suas audiências se sobrepõem e se engajam nas mesmas questões. Se as hashtags são muito diferentes, as duas comunidades existem em mundos políticos separados.

---
## 7. Tabela Resumo: Resposta da Comunidade em Um Olhar

Esta tabela reúne todos os números-chave das seções acima em uma visão de fácil comparação.

In [ ]:
# --- Tabela Resumo Comparativa ---
summary_data = []

for user in ['trump', 'aoc']:
    user_replies = replies_df[replies_df['parent_user'] == user]
    user_tweets = df[df['user'] == user]
    n_replies = len(user_replies)
    
    # Total de respostas
    total_replies = n_replies
    
    # Média de respostas por tweet
    avg_replies_per_tweet = user_tweets['reply_count'].mean()
    
    # Mediana do tempo de resposta
    median_delay = user_replies['reply_delay_minutes'].median()
    
    # Porcentagens de sentimento
    sent_counts = user_replies['sentiment_simple'].value_counts()
    pct_positive = (sent_counts.get('positive', 0) / n_replies * 100) if n_replies > 0 else 0
    pct_negative = (sent_counts.get('negative', 0) / n_replies * 100) if n_replies > 0 else 0
    
    # Média de seguidores dos respondentes
    mean_followers = user_replies['user_followers_count'].mean()
    
    # % respondentes verificados (legado ou azul)
    pct_verified = (
        (user_replies['user_is_verified'].sum() + user_replies['user_is_blue_verified'].sum())
        / n_replies * 100
    ) if n_replies > 0 else 0
    
    # Resposta mais curtida
    if n_replies > 0:
        top_reply = user_replies.loc[user_replies['favorite_count'].idxmax()]
        most_liked_text = str(top_reply['text'])[:60] + '...'
        most_liked_likes = int(top_reply['favorite_count'])
    else:
        most_liked_text = 'N/A'
        most_liked_likes = 0
    
    summary_data.append({
        'Métrica': get_user_label(user),
        'Total de Comentários': f"{total_replies:,}",
        'Média Comentários por Tweet': f"{avg_replies_per_tweet:,.0f}",
        'Tempo Típico de Resposta (min)': f"{median_delay:,.1f}",
        '% Comentários Positivos': f"{pct_positive:.1f}%",
        '% Comentários Negativos': f"{pct_negative:.1f}%",
        'Média Seguidores Comentaristas': f"{mean_followers:,.0f}",
        '% Comentaristas Verificados': f"{pct_verified:.1f}%",
        f'Comentário Mais Curtido ({most_liked_likes:,} curtidas)': most_liked_text,
    })

summary_df = pd.DataFrame(summary_data).set_index('Métrica').T
summary_df.index.name = 'Métrica'

styled = summary_df.style.set_caption(
    'Comparação de Resposta da Comunidade: Trump vs AOC'
).set_table_styles([
    {'selector': 'caption', 'props': [('font-size', '14px'), ('font-weight', 'bold')]},
    {'selector': 'th', 'props': [('text-align', 'left')]},
    {'selector': 'td', 'props': [('text-align', 'center')]},
])

display(styled)

---
## 8. Principais Conclusões: O Que as Respostas da Comunidade Nos Dizem Sobre o Discurso Político

Após analisar milhares de comentários nos tweets de Trump e AOC, aqui está o que encontramos:

**1. A escala de conversa difere dramaticamente.** O volume absoluto de respostas que cada político recebe reflete suas diferentes posições no cenário político. O político com mais respostas não é necessariamente mais popular -- pode simplesmente ser mais controverso, gerando tanto apoio quanto oposição.

**2. A velocidade de resposta revela dedicação da audiência.** O tempo que leva para os comentários aparecerem nos diz quão de perto cada audiência monitora seu político. Respostas mais rápidas sugerem um público mais devoto e em tempo real que ativamente busca por conteúdo novo, enquanto respostas mais lentas indicam uma audiência mais casual que encontra tweets por recomendação algorítmica.

**3. As seções de comentários são dominadas por pessoas comuns, não por elites.** A vasta maioria dos comentaristas tem contagens de seguidores relativamente modestas, mostrando que a discussão política no Twitter/X é movida por cidadãos comuns e não por figuras midiáticas ou celebridades. No entanto, o pequeno número de contas de muitos seguidores que participam pode influenciar desproporcionalmente a conversa.

**4. O tom emocional dos comentários revela o caráter de cada comunidade.** O equilíbrio entre comentários positivos, negativos e neutros nos diz se a seção de respostas de um político funciona mais como uma comunidade de apoio ou como um campo de batalha contestado. Nenhum padrão é inerentemente melhor -- ambos refletem como cidadãos participam de discussões políticas online.

**5. O que é recompensado importa tanto quanto o que é dito.** Os tipos de comentários que recebem mais curtidas revelam o que a audiência mais ampla valoriza. Se comentários críticos são recompensados com curtidas, vozes de oposição são amplificadas. Se comentários de apoio são recompensados, a comunidade reforça a lealdade.

**6. Escolhas de palavras e hashtags revelam mundos políticos separados.** As palavras e hashtags mais comuns em cada seção de comentários mostram que as audiências de Trump e AOC podem estar discutindo tópicos completamente diferentes, refletindo a polarização da conversa política em espaços digitais.

**7. As seções de respostas se tornaram uma arena política por direito próprio.** Quando os próprios comentários recebem milhares de curtidas e repostagens, a seção de comentários deixa de ser apenas uma reação ao tweet original -- se torna um espaço independente para expressão política, debate e mobilização. Isso mostra como as redes sociais transformaram a comunicação política de uma transmissão unilateral em uma conversa multicamadas.

---
*Estas descobertas complementam as análises de conteúdo e engajamento dos notebooks anteriores, construindo um quadro abrangente do discurso político digital.*